### Architecture
```
  XLM-RoBERTa → Token embeddings [B, T, 768]
                    │
                    ├──→ [CLS] repr ────────────────────→ Branch A
                    │
                    ├──→ Lang Embed → Conditioned
                    │    [B, 32]      Attention ─────────→ Branch B
                    │
                    └──→ Anchor Head → Sparse token
                         (per-token)   importance ───────→ Branch C
                              │
                         Sparsity loss (entropy reg)
                    
                    Concat(A, B, C) + Lang Embed → Classifier
```

In [1]:
import os
import warnings
from pathlib import Path
from typing import List
from dataclasses import dataclass, field
from collections import defaultdict
import json

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.cuda.amp import GradScaler, autocast

from transformers import XLMRobertaModel, XLMRobertaTokenizer

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
# if torch.cuda.is_available():
#     print(f"GPU: {torch.cuda.get_device_name(0)}")
#     print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

Device: cuda


## Configuration

In [ ]:
@dataclass
class Config:
    base_path: str = "/workspace/ECHO"
    test_path: str = "/workspace/ECHO_Test"
    output_path: str = "/workspace/outputs_lcts"

    languages: List[str] = field(default_factory=lambda: [
        "bengali", "chinese", "hindi", "english", "french",
        "japanese", "malayalam", "tamil", "telugu", "spanish"])

    random_seed: int = 42

    text_model: str = "xlm-roberta-large"
    max_text_length: int = 128
    freeze_text_layers: int = 8

    hidden_dim: int = 128
    dropout: float = 0.3
    language_embed_dim: int = 32

    anchor_sparsity_weight: float = 0.01

    batch_size: int = 32
    epochs: int = 80
    lr_pretrained: float = 2e-5
    lr_custom: float = 3e-4
    weight_decay: float = 0.03
    warmup_epochs: int = 6
    early_stopping_patience: int = 10
    max_grad_norm: float = 1.0

    focal_gamma: float = 2.0
    label_smoothing: float = 0.05

    samples_per_language_per_batch: int = 4

    def __post_init__(self):
        for d in [self.output_path, f"{self.output_path}/models",
                  f"{self.output_path}/results/figures"]:
            os.makedirs(d, exist_ok=True)

config = Config()
print(f"LCTS-Net | Encoder: {config.text_model} | Batch: {config.batch_size}")
print(f"Anchor sparsity: {config.anchor_sparsity_weight}")

LCTS-Net | Encoder: xlm-roberta-large | Batch: 32
Anchor sparsity: 0.01


## Data Preparation

In [3]:
class DataPreparator:
    def __init__(self, cfg):
        self.config = cfg
        self.all_metadata = {}
        self.label_encoder = LabelEncoder()

    @staticmethod
    def normalize_columns(df):
        df.columns = df.columns.str.lower().str.strip()
        for old, new in {'hate_label': 'label', 'hate_speech': 'label', 'hate': 'label',
                         'file_name': 'filename', 'audio_file': 'filename', 'file': 'filename',
                         'text': 'transcription', 'transcript': 'transcription',
                         'sentence': 'transcription'}.items():
            if old in df.columns and new not in df.columns:
                df = df.rename(columns={old: new})
        if 'gender' not in df.columns: df['gender'] = 'unknown'
        if 'transcription' not in df.columns: df['transcription'] = ''
        df['transcription'] = df['transcription'].fillna('').astype(str)
        return df

    def load_all_metadata(self):
        print("Loading metadata...")
        for lang in tqdm(self.config.languages):
            self.all_metadata[lang] = {}
            lp = Path(self.config.base_path) / lang
            for split in ['train', 'validation']:
                sp = lp / split / 'metadata.csv'
                if not sp.exists(): continue
                df = self.normalize_columns(pd.read_csv(sp))
                df['language'] = lang; df['split'] = split
                df = df[df['transcription'].str.strip().str.len() > 0]
                self.all_metadata[lang][split] = df
                print(f"  {lang}/{split}: {len(df)} samples")

    def create_splits(self):
        print("\nCreating splits...")
        train_dfs, val_dfs = [], []
        for lang, splits in self.all_metadata.items():
            for sn, df in splits.items():
                if df.empty: continue
                (train_dfs if sn == 'train' else val_dfs).append(df)
        train_df = pd.concat(train_dfs, ignore_index=True)
        val_df = pd.concat(val_dfs, ignore_index=True)
        all_langs = pd.concat([train_df['language'], val_df['language']])
        self.label_encoder.fit(all_langs)
        for df in [train_df, val_df]:
            df['language_id'] = self.label_encoder.transform(df['language'])
        print(f"  Train: {len(train_df)}, Val: {len(val_df)}")
        return train_df, val_df

## Model

In [ ]:
class LanguageConditionedAttention(nn.Module):
    """Language embedding modulates attention over tokens.
    Different languages have different hate-signaling token patterns."""

    def __init__(self, token_dim, lang_dim, hidden_dim):
        super().__init__()
        self.query = nn.Linear(lang_dim, hidden_dim)
        self.key = nn.Linear(token_dim, hidden_dim)
        self.scale = hidden_dim ** 0.5

    def forward(self, tokens, lang_embed, mask=None):
        q = self.query(lang_embed).unsqueeze(1) 
        k = self.key(tokens)                    
        scores = (q * k).sum(-1) / self.scale 
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e4)
        weights = F.softmax(scores, dim=1)
        return (tokens * weights.unsqueeze(-1)).sum(1), weights


class AnchorDetector(nn.Module):
    def __init__(self, token_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(token_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, tokens, mask=None):
        scores = self.net(tokens).squeeze(-1)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e4)
        weights = F.softmax(scores, dim=1)
        pooled = (tokens * weights.unsqueeze(-1)).sum(1)
        return pooled, weights


def anchor_sparsity_loss(weights, mask=None):
    if mask is not None:
        weights = weights * mask.float()
    entropy = -(weights * (weights + 1e-8).log()).sum(dim=1).mean()
    return entropy


class LCTSNet(nn.Module):

    def __init__(self, num_languages, cfg):
        super().__init__()
        self.config = cfg
        dim = cfg.hidden_dim
        lang_dim = cfg.language_embed_dim

        # Encoder
        print(f"Loading {cfg.text_model}...")
        self.encoder = XLMRobertaModel.from_pretrained(cfg.text_model)
        text_dim = self.encoder.config.hidden_size  # 768

        for p in self.encoder.embeddings.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.encoder.encoder.layer):
            if i < cfg.freeze_text_layers:
                for p in layer.parameters():
                    p.requires_grad = False
        n_total = len(self.encoder.encoder.layer)
        print(f"  {text_dim}d, {cfg.freeze_text_layers}/{n_total} frozen")

        # Language embedding
        self.lang_embed = nn.Embedding(num_languages, lang_dim)

        # Branch A: [CLS]
        self.cls_proj = nn.Sequential(
            nn.Linear(text_dim, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(cfg.dropout))

        # Branch B: Language-conditioned attention
        self.lang_attn = LanguageConditionedAttention(text_dim, lang_dim, dim)
        self.lang_proj = nn.Sequential(
            nn.Linear(text_dim, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(cfg.dropout))

        # Branch C: Anchor detection
        self.anchor = AnchorDetector(text_dim, dim)
        self.anchor_proj = nn.Sequential(
            nn.Linear(text_dim, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(cfg.dropout))

        # Classifier
        clf_in = dim * 3 + lang_dim
        self.classifier = nn.Sequential(
            nn.Linear(clf_in, dim), nn.GELU(), nn.Dropout(cfg.dropout),
            nn.Linear(dim, dim // 2), nn.GELU(), nn.Dropout(cfg.dropout * 0.5),
            nn.Linear(dim // 2, 2))

    def forward(self, input_ids, attention_mask, language_ids):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        tokens = out.last_hidden_state  

        lang = self.lang_embed(language_ids)

        # Branch A
        branch_a = self.cls_proj(tokens[:, 0])

        # Branch B
        lang_pooled, lang_weights = self.lang_attn(tokens, lang, attention_mask)
        branch_b = self.lang_proj(lang_pooled)

        # Branch C
        anchor_pooled, anchor_weights = self.anchor(tokens, attention_mask)
        branch_c = self.anchor_proj(anchor_pooled)

        combined = torch.cat([branch_a, branch_b, branch_c, lang], dim=1)
        logits = self.classifier(combined)

        return {
            'logits': logits,
            'anchor_weights': anchor_weights,
            'lang_weights': lang_weights,
        }

## Dataset & DataLoader

In [5]:
class TextDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.texts = self.df['transcription'].values
        self.labels = self.df['label'].values.astype(np.int64)
        self.lang_ids = self.df['language_id'].values.astype(np.int64)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), max_length=self.max_len,
                             padding='max_length', truncation=True, return_tensors='pt')
        return (enc['input_ids'].squeeze(0), enc['attention_mask'].squeeze(0),
                self.labels[idx], self.lang_ids[idx])


class LanguageBalancedSampler(Sampler):
    def __init__(self, lang_ids, labels, spl=4, nb=None):
        ul = np.unique(lang_ids); self.nl = len(ul)
        self.lci = {l: {c: np.where((lang_ids == l) & (labels == c))[0] for c in [0, 1]} for l in ul}
        self.spl = spl; self.bs = spl * self.nl
        self.nb = nb or max(1, len(lang_ids) // self.bs)

    def __iter__(self):
        idx = []
        for _ in range(self.nb):
            for l in self.lci:
                for i in range(self.spl):
                    c = i % 2; pool = self.lci[l][c]
                    if len(pool) == 0: pool = self.lci[l][1 - c]
                    if len(pool) > 0: idx.append(np.random.choice(pool))
        return iter(idx)

    def __len__(self):
        return self.nb * self.bs


def create_data_loaders(train_df, val_df, tokenizer, cfg):
    trd = TextDataset(train_df, tokenizer, cfg.max_text_length)
    vd = TextDataset(val_df, tokenizer, cfg.max_text_length)
    sam = LanguageBalancedSampler(
        train_df['language_id'].values, train_df['label'].values,
        cfg.samples_per_language_per_batch)
    return (
        DataLoader(trd, batch_size=cfg.batch_size, sampler=sam,
                   num_workers=2, pin_memory=True, drop_last=True),
        DataLoader(vd, batch_size=cfg.batch_size, shuffle=False,
                   num_workers=2, pin_memory=True))

## Loss & Training

In [6]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, smoothing=0.05):
        super().__init__()
        self.gamma = gamma; self.smoothing = smoothing

    def forward(self, inputs, targets):
        nc = inputs.size(1)
        with torch.no_grad():
            sm = torch.zeros_like(inputs).scatter_(1, targets.unsqueeze(1), 1.0)
            sm = sm * (1 - self.smoothing) + self.smoothing / nc
        lp = F.log_softmax(inputs, dim=1); p = torch.exp(lp)
        return (-(sm * (1 - p) ** self.gamma * lp).sum(1)).mean()


class Trainer:
    def __init__(self, model, cfg, device):
        self.model = model; self.config = cfg; self.device = device

        pre, cust = [], []
        for n, p in model.named_parameters():
            if not p.requires_grad: continue
            (pre if n.startswith('encoder.') else cust).append(p)

        self.optimizer = optim.AdamW([
            {'params': pre, 'lr': cfg.lr_pretrained},
            {'params': cust, 'lr': cfg.lr_custom},
        ], weight_decay=cfg.weight_decay)

        self.criterion = FocalLoss(cfg.focal_gamma, cfg.label_smoothing)
        self.scaler = GradScaler()
        self.scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            self.optimizer, T_0=10, T_mult=2, eta_min=1e-6)

        self.history = defaultdict(list)
        self.best_f1 = 0; self.patience = 0

    def warmup_lr(self, epoch):
        if epoch < self.config.warmup_epochs:
            f = (epoch + 1) / self.config.warmup_epochs
            self.optimizer.param_groups[0]['lr'] = self.config.lr_pretrained * f
            self.optimizer.param_groups[1]['lr'] = self.config.lr_custom * f

    def train_epoch(self, loader, epoch):
        self.model.train()
        tot, tot_cls, tot_sp = 0, 0, 0
        ap, al = [], []

        for ids, mask, lab, lang in tqdm(loader, desc=f"Train E{epoch+1}"):
            ids, mask = ids.to(self.device), mask.to(self.device)
            lab, lang = lab.to(self.device), lang.to(self.device)

            self.optimizer.zero_grad()
            with autocast():
                out = self.model(ids, mask, lang)
                cls_loss = self.criterion(out['logits'], lab)
                sp_loss = anchor_sparsity_loss(out['anchor_weights'], mask)
                loss = cls_loss + self.config.anchor_sparsity_weight * sp_loss

            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.max_grad_norm)
            self.scaler.step(self.optimizer)
            self.scaler.update()

            tot += loss.item(); tot_cls += cls_loss.item(); tot_sp += sp_loss.item()
            al.extend(lab.cpu().numpy())
            ap.extend(out['logits'].argmax(1).cpu().numpy())

        n = max(len(loader), 1)
        return {'loss': tot / n, 'cls_loss': tot_cls / n, 'sparsity_loss': tot_sp / n,
                'f1': f1_score(al, ap, zero_division=0), 'acc': accuracy_score(al, ap)}

    @torch.no_grad()
    def validate(self, loader):
        self.model.eval()
        tot = 0; ap, al, aprob = [], [], []

        for ids, mask, lab, lang in loader:
            ids, mask = ids.to(self.device), mask.to(self.device)
            lab, lang = lab.to(self.device), lang.to(self.device)
            out = self.model(ids, mask, lang)
            tot += self.criterion(out['logits'], lab).item()
            aprob.extend(F.softmax(out['logits'], 1)[:, 1].cpu().numpy())
            ap.extend(out['logits'].argmax(1).cpu().numpy())
            al.extend(lab.cpu().numpy())

        try: auc = roc_auc_score(al, aprob)
        except: auc = 0.5
        return {'loss': tot / max(len(loader), 1),
                'f1': f1_score(al, ap, zero_division=0),
                'precision': precision_score(al, ap, zero_division=0),
                'recall': recall_score(al, ap, zero_division=0),
                'acc': accuracy_score(al, ap), 'auc': auc}

    def train(self, tl, vl):
        print(f"\n{'='*60}\nTRAINING LCTS-Net\n{'='*60}")
        for epoch in range(self.config.epochs):
            self.warmup_lr(epoch)
            tm = self.train_epoch(tl, epoch)
            vm = self.validate(vl)
            if epoch >= self.config.warmup_epochs:
                self.scheduler.step(epoch - self.config.warmup_epochs)

            gap = tm['f1'] - vm['f1']
            lr = self.optimizer.param_groups[1]['lr']
            print(f"\nEpoch {epoch+1}/{self.config.epochs} | LR: {lr:.2e}")
            print(f"  Train — cls: {tm['cls_loss']:.4f}, sparsity: {tm['sparsity_loss']:.4f}, "
                  f"F1: {tm['f1']:.4f}, Acc: {tm['acc']:.4f}")
            print(f"  Val   — loss: {vm['loss']:.4f}, F1: {vm['f1']:.4f}, Acc: {vm['acc']:.4f}, "
                  f"P: {vm['precision']:.3f}, R: {vm['recall']:.3f}, AUC: {vm['auc']:.4f}")
            print(f"  Gap: {gap:.4f}")

            for k, v in tm.items(): self.history[f'train_{k}'].append(v)
            for k, v in vm.items(): self.history[f'val_{k}'].append(v)

            if vm['f1'] > self.best_f1:
                self.best_f1 = vm['f1']; self.patience = 0
                torch.save({'epoch': epoch, 'model_state_dict': self.model.state_dict(),
                             'val_f1': self.best_f1},
                           f"{self.config.output_path}/models/best_model.pt")
                print(f"  ✓ New best (F1: {self.best_f1:.4f})")
            else:
                self.patience += 1
            if self.patience >= self.config.early_stopping_patience:
                print(f"\nEarly stopping at epoch {epoch+1}"); break

        ckpt = f"{self.config.output_path}/models/best_model.pt"
        if os.path.exists(ckpt):
            s = torch.load(ckpt, map_location=self.device)
            self.model.load_state_dict(s['model_state_dict'])
            print(f"\nLoaded best from epoch {s['epoch']+1}")
        return self.model, dict(self.history)

## Validation Thresholds & Inference

In [7]:
def compute_val_thresholds(model, val_loader, val_df, device):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for ids, mask, lab, lang in val_loader:
            ids, mask, lang = ids.to(device), mask.to(device), lang.to(device)
            out = model(ids, mask, lang)
            all_probs.extend(F.softmax(out['logits'], 1)[:, 1].cpu().numpy())
            all_labels.extend(lab.numpy())

    all_probs, all_labels = np.array(all_probs), np.array(all_labels)
    languages = val_df['language'].values[:len(all_labels)]

    thresholds = {}
    for lang in np.unique(languages):
        m = languages == lang
        if m.sum() > 5:
            best_t, best_f1 = 0.5, 0
            for t in np.arange(0.25, 0.75, 0.02):
                f = f1_score(all_labels[m], (all_probs[m] >= t).astype(int), zero_division=0)
                if f > best_f1: best_f1, best_t = f, t
            thresholds[lang] = best_t
            print(f"  {lang:12s}: τ={best_t:.2f} (F1={best_f1:.3f})")
        else:
            thresholds[lang] = 0.5
    return thresholds


def run_inference(model, tokenizer, config, label_encoder, thresholds):
    model.eval()
    test_path = Path(config.test_path)
    print(f"\n{'='*60}\nINFERENCE on {test_path}\n{'='*60}")

    for lang in config.languages:
        meta_path = test_path / lang / 'metadata.csv'
        if not meta_path.exists():
            print(f"  {lang}: skipping"); continue

        df = pd.read_csv(meta_path)
        df.columns = df.columns.str.lower().str.strip()
        for old, new in {'text': 'transcription', 'transcript': 'transcription',
                         'sentence': 'transcription'}.items():
            if old in df.columns and new not in df.columns:
                df = df.rename(columns={old: new})
        text_col = 'transcription' if 'transcription' in df.columns else 'text'
        df[text_col] = df[text_col].fillna('').astype(str)

        lang_id = label_encoder.transform([lang])[0]
        thresh = thresholds.get(lang, 0.5)

        all_probs = []
        for start in range(0, len(df), config.batch_size):
            texts = list(df[text_col].values[start:start + config.batch_size])
            enc = tokenizer(texts, max_length=config.max_text_length,
                           padding='max_length', truncation=True, return_tensors='pt')
            ids, mask = enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE)
            lang_ids = torch.full((ids.size(0),), lang_id, dtype=torch.long, device=DEVICE)

            with torch.no_grad():
                out = model(ids, mask, lang_ids)
                all_probs.extend(F.softmax(out['logits'], 1)[:, 1].cpu().numpy())

        df['label'] = (np.array(all_probs) >= thresh).astype(int)
        out_path = f"{config.output_path}/{lang}.csv"
        df.to_csv(out_path, index=False)
        print(f"  {lang:12s}: {len(df)} samples, τ={thresh:.2f}, "
              f"{df['label'].mean()*100:.1f}% hate → {out_path}")

## Pipeline

In [8]:
def run_pipeline(config):
    print("=" * 70)
    print("LCTS-Net (Text-Only)")
    print("=" * 70)

    prep = DataPreparator(config)
    prep.load_all_metadata()
    train_df, val_df = prep.create_splits()

    print(f"\nLoading tokenizer: {config.text_model}")
    tokenizer = XLMRobertaTokenizer.from_pretrained(config.text_model)
    train_loader, val_loader = create_data_loaders(train_df, val_df, tokenizer, config)

    model = LCTSNet(num_languages=len(config.languages), cfg=config).to(DEVICE)
    nt = sum(p.numel() for p in model.parameters())
    nr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Params: {nt:,} total, {nr:,} trainable ({nr/nt*100:.1f}%)")

    trainer = Trainer(model, config, DEVICE)
    model, history = trainer.train(train_loader, val_loader)
    with open(f"{config.output_path}/models/history.json", 'w') as f:
        json.dump(history, f, indent=2)

    print("\nComputing per-language thresholds on validation...")
    thresholds = compute_val_thresholds(model, val_loader, val_df, DEVICE)

    run_inference(model, tokenizer, config, prep.label_encoder, thresholds)

    return model, history

In [9]:
if not os.path.exists(config.base_path):
    print(f"Dataset not found at {config.base_path}")
else:
    model, history = run_pipeline(config)

LCTS-Net (Text-Only)
Loading metadata...


  0%|          | 0/10 [00:00<?, ?it/s]

  bengali/train: 1096 samples
  bengali/validation: 200 samples
  chinese/train: 626 samples
  chinese/validation: 134 samples
  hindi/train: 1100 samples
  hindi/validation: 200 samples
  english/train: 1096 samples
  english/validation: 199 samples
  french/train: 767 samples
  french/validation: 164 samples
  japanese/train: 350 samples
  japanese/validation: 75 samples
  malayalam/train: 618 samples
  malayalam/validation: 132 samples
  tamil/train: 356 samples
  tamil/validation: 76 samples
  telugu/train: 385 samples
  telugu/validation: 82 samples
  spanish/train: 870 samples
  spanish/validation: 186 samples

Creating splits...
  Train: 7264, Val: 1448

Loading tokenizer: xlm-roberta-large


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...


Epoch 34/80 | LR: 1.73e-05
  Train — cls: 0.0361, sparsity: 0.0046, F1: 0.9978, Acc: 0.9978
  Val   — loss: 0.1613, F1: 0.8192, Acc: 0.8494, P: 0.837, R: 0.802, AUC: 0.8912
  Gap: 0.1786


Train E35:   0%|          | 0/226 [00:00<?, ?it/s]


Epoch 35/80 | LR: 8.32e-06
  Train — cls: 0.0351, sparsity: 0.0071, F1: 0.9990, Acc: 0.9990
  Val   — loss: 0.1664, F1: 0.8141, Acc: 0.8474, P: 0.845, R: 0.786, AUC: 0.8877
  Gap: 0.1849


Train E36:   0%|          | 0/226 [00:00<?, ?it/s]


Epoch 36/80 | LR: 2.84e-06
  Train — cls: 0.0351, sparsity: 0.0068, F1: 0.9994, Acc: 0.9994
  Val   — loss: 0.1687, F1: 0.8101, Acc: 0.8446, P: 0.844, R: 0.779, AUC: 0.8830
  Gap: 0.1893

Early stopping at epoch 36

Loaded best from epoch 26

Computing per-language thresholds on validation...
  bengali     : τ=0.63 (F1=0.849)
  chinese     : τ=0.25 (F1=0.660)
  english     : τ=0.31 (F1=0.878)
  french      : τ=0.53 (F1=0.821)
  hindi       : τ=0.57 (F1=0.814)
  japanese    : τ=0.25 (F1=1.000)
  malayalam   : τ=0.25 (F1=0.957)
  spanish     : τ=0.31 (F1=0.632)
  tamil       : τ=0.25 (F1=0.847)
  telugu      : τ=0.25 (F1=0.943)

INFERENCE on /workspace/ECHO_Test
  bengali     : 200 samples, τ=0.63, 47.0% hate → /workspace/outputs_lcts/bengali.csv
  chinese     : 136 samples, τ=0.25, 32.4% hate → /workspace/outputs_lcts/chinese.csv
  hindi       : 200 samples, τ=0.57, 15.5% hate → /workspace/outputs_lcts/hindi.csv
  english     : 199 samples, τ=0.31, 48.2% hate → /workspace/outputs_lcts/